## 1. Import Libraries

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
import lightgbm as lgb
from sklearn.feature_selection import RFE
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score

## 2. Data Collection and understanding

In [25]:
credit_data = pd.read_csv("credit_card_clean.csv")
credit_data

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT
0,1,20000.0,female,university,married,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,female,university,single,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,female,university,single,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,female,university,married,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,male,university,married,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,220000.0,male,highschool,married,39,0,0,0,0,...,88004.0,31237.0,15980.0,8500.0,20000.0,5003.0,3047.0,5000.0,1000.0,0
29996,29997,150000.0,male,highschool,single,43,-1,-1,-1,-1,...,8979.0,5190.0,0.0,1837.0,3526.0,8998.0,129.0,0.0,0.0,0
29997,29998,30000.0,male,university,single,37,4,3,2,-1,...,20878.0,20582.0,19357.0,0.0,0.0,22000.0,4200.0,2000.0,3100.0,1
29998,29999,80000.0,male,highschool,married,41,1,-1,0,0,...,52774.0,11855.0,48944.0,85900.0,3409.0,1178.0,1926.0,52964.0,1804.0,1


In [11]:
credit_data.shape

(30000, 25)

In [12]:
credit_data.isnull().sum()

ID           0
LIMIT_BAL    0
SEX          0
EDUCATION    0
MARRIAGE     0
AGE          0
PAY_1        0
PAY_2        0
PAY_3        0
PAY_4        0
PAY_5        0
PAY_6        0
BILL_AMT1    0
BILL_AMT2    0
BILL_AMT3    0
BILL_AMT4    0
BILL_AMT5    0
BILL_AMT6    0
PAY_AMT1     0
PAY_AMT2     0
PAY_AMT3     0
PAY_AMT4     0
PAY_AMT5     0
PAY_AMT6     0
DEFAULT      0
dtype: int64

In [13]:
credit_data.dtypes

ID             int64
LIMIT_BAL    float64
SEX           object
EDUCATION     object
MARRIAGE      object
AGE            int64
PAY_1          int64
PAY_2          int64
PAY_3          int64
PAY_4          int64
PAY_5          int64
PAY_6          int64
BILL_AMT1    float64
BILL_AMT2    float64
BILL_AMT3    float64
BILL_AMT4    float64
BILL_AMT5    float64
BILL_AMT6    float64
PAY_AMT1     float64
PAY_AMT2     float64
PAY_AMT3     float64
PAY_AMT4     float64
PAY_AMT5     float64
PAY_AMT6     float64
DEFAULT        int64
dtype: object

## 3. Data Cleaning and Preprocessing

In [26]:
## Drop ID column

del credit_data["ID"]

In [28]:
label_encoder = LabelEncoder()

credit_data["SEX"] = label_encoder.fit_transform(credit_data["SEX"])
credit_data["EDUCATION"] = label_encoder.fit_transform(credit_data["EDUCATION"])
credit_data["MARRIAGE"] = label_encoder.fit_transform(credit_data["MARRIAGE"])

credit_data

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT
0,20000.0,0,3,0,24,2,2,-1,-1,-2,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,120000.0,0,3,2,26,-1,2,0,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,90000.0,0,3,2,34,0,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,50000.0,0,3,0,37,0,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,50000.0,1,3,0,57,-1,0,-1,0,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000.0,1,1,0,39,0,0,0,0,0,...,88004.0,31237.0,15980.0,8500.0,20000.0,5003.0,3047.0,5000.0,1000.0,0
29996,150000.0,1,1,2,43,-1,-1,-1,-1,0,...,8979.0,5190.0,0.0,1837.0,3526.0,8998.0,129.0,0.0,0.0,0
29997,30000.0,1,3,2,37,4,3,2,-1,0,...,20878.0,20582.0,19357.0,0.0,0.0,22000.0,4200.0,2000.0,3100.0,1
29998,80000.0,1,1,0,41,1,-1,0,0,0,...,52774.0,11855.0,48944.0,85900.0,3409.0,1178.0,1926.0,52964.0,1804.0,1


In [35]:
## Split the dataset into X input and y output

X = credit_data.drop("DEFAULT", axis =1)
y = credit_data["DEFAULT"]

In [36]:
## Split the data into training and testing data

X_train,X_test,y_train,y_test = train_test_split(X,y,shuffle= True,stratify=y,random_state=123)

## 4. Model Building, Training, Testing

In [33]:
def runRandomForestClassifier(X_train,X_test,y_train,y_test):
    ## Model Building
    rfe_classifier = RandomForestClassifier(max_depth=5, random_state=123)
    rfe_classifier.fit(X_train,y_train)
    
    ## Model Training
    rfe_classifier.predict(X_train)
    
    # Model Testing
    y_pred_test = rfe_classifier.predict(X_test)
    
    ## Model Evaluation
    
    print("Confusion matrix\n", confusion_matrix(y_test,y_pred_test))
    print("Accuracy Score\t", accuracy_score(y_test,y_pred_test))
    print("Precision Score\t", precision_score(y_test,y_pred_test))
    print("Recall Score\t", recall_score(y_test,y_pred_test))

In [34]:
runRandomForestClassifier(X_train,X_test,y_train,y_test)

Confusion matrix
 [[5652  189]
 [1211  448]]
Accuracy Score	 0.8133333333333334
Precision Score	 0.7032967032967034
Recall Score	 0.270042194092827


## Lets use **feature selection** to select the optimal features for model Building

In [37]:

rfe_model = RFE(estimator = GradientBoostingClassifier(random_state=123), n_features_to_select = 1 )
rfe_model.fit(X_train,y_train)

,estimator,GradientBoostingClassifier()
,n_features_to_select,1
,step,1
,verbose,0
,importance_getter,'auto'
,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2


In [43]:
for i in range(1,25):
    rfe_model = RFE(estimator = GradientBoostingClassifier(random_state=123), n_features_to_select = i)
    rfe_model.fit(X_train,y_train)
            
    X_train_rfe = rfe_model.transform(X_train)
    X_test_rfe = rfe_model.transform(X_test)
    print("Selected Feature:",i)
        
    runRandomForestClassifier(X_train_rfe,X_test_rfe,y_train,y_test)
    print("*************************")  

Selected Feature: 1
Confusion matrix
 [[5627  214]
 [1117  542]]
Accuracy Score	 0.8225333333333333
Precision Score	 0.716931216931217
Recall Score	 0.32670283303194697
*************************
Selected Feature: 2
Confusion matrix
 [[5624  217]
 [1112  547]]
Accuracy Score	 0.8228
Precision Score	 0.7159685863874345
Recall Score	 0.3297166968053044
*************************
Selected Feature: 3
Confusion matrix
 [[5629  212]
 [1115  544]]
Accuracy Score	 0.8230666666666666
Precision Score	 0.7195767195767195
Recall Score	 0.32790837854128996
*************************
Selected Feature: 4
Confusion matrix
 [[5627  214]
 [1108  551]]
Accuracy Score	 0.8237333333333333
Precision Score	 0.7202614379084967
Recall Score	 0.33212778782399033
*************************
Selected Feature: 5
Confusion matrix
 [[5631  210]
 [1115  544]]
Accuracy Score	 0.8233333333333334
Precision Score	 0.7214854111405835
Recall Score	 0.32790837854128996
*************************
Selected Feature: 6
Confusion matr

C:\Users\Madhumitha B\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\feature_selection\_rfe.py:300: UserWarning: Found n_features_to_select=24 > n_features=23. There will be no feature selection and all features will be kept.
  warnings.warn(


Selected Feature: 24
Confusion matrix
 [[5643  198]
 [1207  452]]
Accuracy Score	 0.8126666666666666
Precision Score	 0.6953846153846154
Recall Score	 0.27245328511151296
*************************


## To determine the optimal subset of features that maximizes the classification performance of a Random Forest model using Recursive Feature Elimination (RFE) with Gradient Boosting.

In [ ]:
best_score = 0
best_features = []

for i in range(1, 25):
    rfe_model = RFE(
        estimator=GradientBoostingClassifier(random_state=42),
        n_features_to_select=i,
    )
    rfe_model.fit(X_train, y_train)

    X_train_rfe = rfe_model.transform(X_train)
    X_test_rfe = rfe_model.transform(X_test)

    print(f"Selected Feature Count: {i}")

    # Ensure your function returns accuracy/score so you can track the best run
    current_acc = runRandomForestClassifier(
        X_train_rfe, X_test_rfe, y_train, y_test
    )
    print("***********************************")

    # Keep track of the feature subset that produced the highest score
    if current_acc is not None and current_acc > best_score:
        best_score = current_acc
        best_features = X.columns[rfe_model.get_support()].tolist()

# Printed once AFTER all 24 loops complete:
print("\n--- FINAL BEST SELECTION ---")
print("Best Score:", best_score)
print("Count of Selected Features:", len(best_features))
print("Selected Features:", best_features)